<a href="https://colab.research.google.com/github/kartikigaikwad/Amazon-Clone/blob/main/kartiki_App_B.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -U openai==1.55.3 langchain langchain-openai langchain-core pinecone gradio arxiv tqdm

import arxiv
import openai
import pinecone
import os
from tqdm import tqdm
from google.colab import userdata
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_openai import AzureOpenAIEmbeddings
from pinecone import Pinecone, ServerlessSpec, PineconeApiException
import gradio as gr

# Load secrets from Colab userdata
def _set_env(var: str):
    if not os.environ.get(var):
        os.environ[var] = userdata.get(f"{var}")

for var in ["AZURE_OPENAI_API_KEY", "AZURE_OPENAI_ENDPOINT", "USER_AGENT",
            "PINECONE_API_KEY", "PINECONE_INDEX", "OPENAI_API_VERSION"]:
    _set_env(var)

openai.api_key = os.getenv("AZURE_OPENAI_API_KEY")
openai.api_base = os.getenv("AZURE_OPENAI_ENDPOINT")
openai.api_type = "azure"
openai.api_version = os.getenv("OPENAI_API_VERSION")

pinecone_api_key = os.getenv("PINECONE_API_KEY")
pinecone_index_name = os.getenv("PINECONE_INDEX")

pinecone_client = Pinecone(api_key=pinecone_api_key)

# Create index if not present
try:
    existing_indexes = [i['name'] for i in pinecone_client.list_indexes()]
    if pinecone_index_name not in existing_indexes:
        print(f"Creating new Pinecone index: {pinecone_index_name}")
        pinecone_client.create_index(
            name=pinecone_index_name,
            dimension=1536,
            metric="cosine",
            spec=ServerlessSpec(cloud='aws', region='us-east-1')
        )
    else:
        print(f"Pinecone index '{pinecone_index_name}' already exists.")
except PineconeApiException as e:
    if e.status == 409 and "ALREADY_EXISTS" in e.body:
        print(f"Pinecone index '{pinecone_index_name}' already exists.")
    else:
        raise

index = pinecone_client.Index(pinecone_index_name)
print(f" Connected to Pinecone index: {pinecone_index_name}")


  Using cached langchain-1.0.8-py3-none-any.whl.metadata (4.9 kB)
  Using cached langchain_openai-1.0.3-py3-none-any.whl.metadata (2.6 kB)
  Using cached langchain_core-1.1.0-py3-none-any.whl.metadata (3.6 kB)
  Using cached langgraph-1.0.3-py3-none-any.whl.metadata (7.8 kB)
INFO: pip is looking at multiple versions of langchain-openai to determine which version is compatible with other requirements. This could take a while.
  Using cached langchain_openai-1.0.2-py3-none-any.whl.metadata (1.8 kB)
  Using cached langchain_openai-1.0.1-py3-none-any.whl.metadata (1.8 kB)
  Using cached langchain_openai-1.0.0-py3-none-any.whl.metadata (1.8 kB)
  Using cached langchain_openai-0.3.35-py3-none-any.whl.metadata (2.4 kB)
  Using cached langchain_openai-0.3.34-py3-none-any.whl.metadata (2.4 kB)
  Using cached langchain_openai-0.3.33-py3-none-any.whl.metadata (2.4 kB)
  Using cached langchain_openai-0.3.32-py3-none-any.whl.metadata (2.4 kB)
INFO: pip is still looking at multiple versions of langc

In [ ]:
import uuid
import json
from typing import List

def search_arxiv(query: str, max_results: int = 5):
    print(f"\n🔍 Searching arXiv for '{query}' ...")
    search = arxiv.Search(query=query, max_results=max_results, sort_by=arxiv.SortCriterion.Relevance)
    results = []
    for r in search.results():
        results.append({
            "id": r.get_short_id(),
            "title": r.title,
            "authors": [a.name for a in r.authors],
            "summary": r.summary,
            "pdf_url": r.pdf_url,
            "published": r.published.isoformat() if r.published else None
        })
    print(f"✅ Found {len(results)} papers.")
    return results

def chunk_texts(texts: List[str], chunk_size: int = 1000, overlap: int = 100):
    splitter = RecursiveCharacterTextSplitter(chunk_size=chunk_size, chunk_overlap=overlap)
    chunks = splitter.split_text("\n".join(texts))
    print(f"🧩 Created {len(chunks)} text chunks.")
    return chunks

def get_embeddings_client(deployment="text-embedding-3-small"):
    return AzureOpenAIEmbeddings(deployment=deployment)

def embed_texts(emb_client, texts: List[str], batch_size: int = 8):
    print("⚙️ Generating embeddings...")
    embeddings = []
    for i in tqdm(range(0, len(texts), batch_size)):
        batch = texts[i:i+batch_size]
        batch_embs = emb_client.embed_documents(batch)
        embeddings.extend(batch_embs)
    print(f"✅ Created embeddings for {len(texts)} chunks.")
    return embeddings

def upsert_to_pinecone(index, embeddings, texts, namespace="default"):
    print(f"🚀 Uploading {len(embeddings)} vectors to Pinecone under namespace '{namespace}'...")
    data = [(str(uuid.uuid4()), emb, {"text": texts[i]}) for i, emb in enumerate(embeddings)]
    index.upsert(vectors=data, namespace=namespace)
    print("✅ Data successfully stored in Pinecone.")


In [ ]:
# ---- Ingest arXiv papers, chunk, embed and upsert ----

def ingest_arxiv_topic_to_pinecone(topic: str, max_results: int = 10,
                                   chunk_size: int = 1000, overlap: int = 200,
                                   namespace: str = "research"):
    """
    1) search arXiv for 'topic'
    2) create text blobs per paper (title + authors + abstract + url + published)
    3) chunk texts
    4) embed chunks
    5) upsert to Pinecone with metadata linking back to the paper
    """
    # 1) search arXiv
    papers = search_arxiv(topic, max_results=max_results)  # uses your function
    if not papers:
        print("No papers found.")
        return []

    # prepare texts and metadata per paper
    texts = []
    metadatas = []
    for p in papers:
        paper_text = f"Title: {p['title']}\nAuthors: {', '.join(p['authors'])}\nPublished: {p['published']}\nURL: {p['pdf_url']}\n\nAbstract:\n{p['summary']}"
        texts.append(paper_text)
        metadatas.append({
            "paper_id": p["id"],
            "title": p["title"],
            "authors": p["authors"],
            "pdf_url": p["pdf_url"],
            "published": p["published"]
        })

    # 2) chunk texts (we will keep association of chunk -> paper metadata)
    splitter = RecursiveCharacterTextSplitter(chunk_size=chunk_size, chunk_overlap=overlap)
    all_chunks = []
    all_chunk_meta = []
    for i, text in enumerate(texts):
        chunks = splitter.split_text(text)
        for ci, c in enumerate(chunks):
            all_chunks.append(c)
            # carry paper-level metadata and chunk index for traceability
            meta = metadatas[i].copy()
            meta.update({"chunk_index": ci})
            all_chunk_meta.append(meta)

    print(f"Prepared {len(all_chunks)} chunks from {len(papers)} papers.")

    # 3) embeddings
    emb_client = get_embeddings_client()  # your helper
    chunk_embeddings = embed_texts(emb_client, all_chunks, batch_size=8)

    # 4) upsert with metadata
    # Generate stable ids so this is idempotent — use uuid with prefix of paper_id and chunk index
    vector_data = []
    for i, emb in enumerate(chunk_embeddings):
        paper_id = all_chunk_meta[i].get("paper_id", "paper")
        chunk_idx = all_chunk_meta[i].get("chunk_index", 0)
        vid = f"{paper_id}__chunk{chunk_idx}__{i}"  # deterministic-ish
        vector_data.append((vid, emb, all_chunk_meta[i]))

    print(f"Upserting {len(vector_data)} vectors to Pinecone namespace='{namespace}' ...")
    index.upsert(vectors=vector_data, namespace=namespace)
    print("Ingestion complete.")
    return papers  # return paper list for UI or records


In [ ]:
# ---- Retrieve relevant chunks and assemble citation mapping ----

def retrieve_chunks_with_metadata(query: str, top_k: int = 5, namespace: str = "research"):
    """
    Returns:
      - retrieved_chunks: list of chunk texts (for LLM context)
      - citations: list of dicts {ref_id, title, authors, pdf_url, score}
    """
    emb_client = get_embeddings_client()
    query_emb = emb_client.embed_query(query)

    # query Pinecone
    res = index.query(
        vector=query_emb,
        top_k=top_k,
        include_metadata=True,
        include_values=False,
        namespace=namespace
    )

    matches = res.get("matches", [])
    if not matches:
        return [], []

    retrieved_chunks = []
    citations_map = {}  # map paper_id -> citation entry with aggregated info

    for i, m in enumerate(matches):
        meta = m.get("metadata", {})
        chunk_text = meta.get("text") or m.get("metadata", {}).get("text", "")  # fallback
        # If we didn't store text under "text", try to fetch from metadata keys
        if not chunk_text:
            # Some of our upserts store full chunk as metadata; if not, skip
            chunk_text = m.get("metadata", {}).get("text", "")

        # If chunk_text still empty, try reading from values (not included above by design)
        retrieved_chunks.append(chunk_text)

        paper_id = meta.get("paper_id", f"unknown_{i}")
        score = m.get("score", None)
        # Add or update citation info
        if paper_id not in citations_map:
            citations_map[paper_id] = {
                "paper_id": paper_id,
                "title": meta.get("title"),
                "authors": meta.get("authors"),
                "pdf_url": meta.get("pdf_url"),
                "published": meta.get("published"),
                "top_score": score,
                "references": [ {"chunk_index": meta.get("chunk_index"), "score": score} ]
            }
        else:
            citations_map[paper_id]["references"].append({"chunk_index": meta.get("chunk_index"), "score": score})
            # keep best (lowest) score or highest similarity depending on Pinecone metric; here we keep max score
            try:
                if score and (citations_map[paper_id]["top_score"] is None or score > citations_map[paper_id]["top_score"]):
                    citations_map[paper_id]["top_score"] = score
            except:
                pass

    # produce citations list ordered by top_score desc
    citations = sorted(citations_map.values(), key=lambda x: (x.get("top_score") is not None, x.get("top_score")), reverse=True)
    return retrieved_chunks, citations


In [ ]:
from openai import AzureOpenAI

# Azure client initialization
client = AzureOpenAI(
    api_key=os.getenv("AZURE_OPENAI_API_KEY"),
    azure_endpoint=os.getenv("AZURE_OPENAI_ENDPOINT"),
    api_version=os.getenv("OPENAI_API_VERSION")
)

AZURE_CHAT_MODEL = "gpt-4.1"   # change if your deployment name is different

def generate_answer_and_citations(query: str, retrieved_chunks: list, citations: list):
    """
    Same functionality — rewritten in your required syntax:
    response = client.chat.completions.create(...)
    """

    # Build context string
    context = "\n\n---\n\n".join(retrieved_chunks)

    # Prepare citations list for system prompt
    citations_text_lines = []
    for idx, c in enumerate(citations, start=1):
        title = c.get("title") or "Untitled"
        authors = ", ".join(c.get("authors") or [])
        pdf = c.get("pdf_url") or c.get("pdf") or ""
        published = c.get("published") or ""
        citations_text_lines.append(
            f"[Ref {idx}] {title} — {authors}. {published}. {pdf}"
        )

    citations_text = "\n".join(citations_text_lines)

    system_prompt = f"""
You are an expert research assistant. Use ONLY the provided context to answer the user's question.
- When you use information from the context, append a citation token like [Ref 1], [Ref 2], etc.
- Do NOT hallucinate. If information is missing, reply: "Insufficient research context."

Context:
{context}

Citations:
{citations_text}
"""

    # --------------------------
    #   YOUR REQUIRED SYNTAX
    # --------------------------
    response = client.chat.completions.create(
        model=AZURE_CHAT_MODEL,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": query}
        ],
        temperature=0.0,
        max_tokens=800,
    )
    # --------------------------

    answer = response.choices[0].message.content

    # Build markdown citation section for the final output
    md_citations = []
    for idx, c in enumerate(citations, start=1):
        title = c.get("title") or "Untitled"
        authors = ", ".join(c.get("authors") or [])
        pdf = c.get("pdf_url") or ""
        published = c.get("published") or ""
        md_line = f"- **[Ref {idx}]** [{title}]({pdf}) — {authors} — {published}"
        md_citations.append(md_line)

    final_output = (
        f"### Answer\n\n{answer}\n\n---\n\n### Cited Papers\n\n" +
        "\n".join(md_citations)
    )
    return final_output


In [ ]:
def handle_research_query(query: str, top_k: int = 5, namespace: str = "research"):
    # 1) retrieve
    retrieved_chunks, citations = retrieve_chunks_with_metadata(query, top_k=top_k, namespace=namespace)
    if not retrieved_chunks:
        return "⚠ No relevant context found in the vectorstore. Consider running the ingestion pipeline (ingest_arxiv_topic_to_pinecone)."

    # 2) generate grounded answer
    result_md = generate_answer_and_citations(query, retrieved_chunks, citations)
    return result_md


In [ ]:
def list_cited_papers_for_query(query: str, top_k: int = 5, namespace: str = "research"):
    _, citations = retrieve_chunks_with_metadata(query, top_k=top_k, namespace=namespace)
    return citations


In [ ]:
import gradio as gr

def gradio_query_fn(query: str, top_k: int = 5, namespace: str = "research"):
    try:
        return handle_research_query(query, top_k=top_k, namespace=namespace)
    except Exception as e:
        return f"Error: {e}"

with gr.Blocks() as demo:
    gr.Markdown("# Research Semantic QA — Grounded Answers with Citations")
    with gr.Row():
        with gr.Column(scale=3):
            txt = gr.Textbox(label="Ask a research question", placeholder="E.g. 'What are the latest transformer-based approaches for protein folding?'")
            top_k_in = gr.Slider(minimum=1, maximum=12, step=1, label="Top K results from vector DB", value=5)
            namespace_in = gr.Textbox(label="Pinecone namespace", value="research")
            btn = gr.Button("Submit")
        with gr.Column(scale=2):
            out = gr.Markdown(label="Grounded Answer + Citations")

    # optional: ingestion controls
    with gr.Accordion("Ingest arXiv topic into Pinecone (optional)", open=False):
        ingest_topic = gr.Textbox(label="ArXiv query/topic", placeholder="E.g. 'graph neural networks'")
        ingest_n = gr.Slider(minimum=1, maximum=100, step=1, label="Max papers to fetch", value=10)
        ingest_btn = gr.Button("Run ingestion")
        ingest_status = gr.Textbox(label="Ingestion status", interactive=False)

    # events
    btn.click(lambda q, k, ns: gradio_query_fn(q, int(k), ns), inputs=[txt, top_k_in, namespace_in], outputs=out)
    def run_ingest(topic, n, ns):
        try:
            papers = ingest_arxiv_topic_to_pinecone(topic, max_results=int(n), namespace=ns)
            return f"Ingested {len(papers)} papers into namespace '{ns}'."
        except Exception as e:
            return f"Ingestion error: {e}"
    ingest_btn.click(run_ingest, inputs=[ingest_topic, ingest_n, namespace_in], outputs=ingest_status)

demo.launch(debug=True)


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://89f23c51a070c1f0c0.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)



🔍 Searching arXiv for 'nlp' ...


/tmp/ipython-input-3072431762.py:9: DeprecationWarning: The 'Search.results' method is deprecated, use 'Client.results' instead
  for r in search.results():


✅ Found 10 papers.
Prepared 27 chunks from 10 papers.
⚙️ Generating embeddings...


100%|██████████| 4/4 [00:01<00:00,  3.26it/s]


✅ Created embeddings for 27 chunks.
Upserting 27 vectors to Pinecone namespace='research' ...
Ingestion complete.
